# Assignment 3 — FlexGen on Colab

Run OPT-1.3B baseline + disk offload + Q1–Q4 on Colab T4.

**Setup**: Runtime → Change runtime type → **GPU (T4)**

> Colab's free T4 has only 12.7 GB RAM, so FlexGen's own download + conversion of OPT-1.3B will OOM. This notebook downloads pre-converted numpy weights from the TA's Drive instead.

## 1. Check GPU and Python version

In [5]:
!nvidia-smi | head -15
!python --version
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory // 1024**2, 'MB')
!free -m | head -2
!df -h /content | head -2

Sun May 24 11:26:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install FlexGen + gdown

Colab's pre-installed PyTorch is usually new enough (2.x + CUDA 12+); no need to reinstall.

In [6]:
%cd /content
![ -d FlexLLMGen ] || git clone https://github.com/FMInference/FlexLLMGen.git
%cd FlexLLMGen
!pip install -q -e .
!pip install -q --upgrade gdown
%cd ..
# Create logs/ and offload directories
!mkdir -p logs flexgen_offload

/content
Cloning into 'FlexLLMGen'...
remote: Enumerating objects: 4553, done.
remote: Total 4553 (delta 0), reused 0 (delta 0), pack-reused 4553 (from 1)
Receiving objects: 100% (4553/4553), 38.11 MiB | 13.78 MiB/s, done.
Resolving deltas: 100% (1320/1320), done.
/content/FlexLLMGen
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for flexllmgen (pyproject.toml) ... done
/content


## 3. Smoke test: OPT-125M (~250 MB, downloads in < 1 minute)

OPT-125M is small enough that FlexGen's own download + conversion won't OOM. Use it to confirm FlexGen is working.

In [7]:
!python -m flexllmgen.flex_opt \
    --model facebook/opt-125m \
    --percent 100 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload

<run_flexllmgen>: args.model: facebook/opt-125m
config.json: 100% 651/651 [00:00<00:00, 3.90MB/s]
tokenizer_config.json: 100% 685/685 [00:00<00:00, 4.70MB/s]
vocab.json: 899kB [00:00, 101MB/s]
merges.txt: 456kB [00:00, 87.2MB/s]
special_tokens_map.json: 100% 221/221 [00:00<00:00, 1.13MB/s]
model size: 0.230 GB, cache size: 0.020 GB, hidden size (prefill): 0.001 GB
init weight...
Load the pre-trained pytorch weights of opt-125m from huggingface. The downloading and cpu loading can take dozens of minutes. If it seems to get stuck, you can monitor the progress by checking the memory usage of this process.
Fetching 1 files: 100% 1/1 [00:06<00:00,  6.29s/it]
Download complete: 100% 251M/251M [00:06<00:00, 174MB/s]                
Download complete: 100% 251M/251M [00:06<00:00, 38.9MB/s]
                                      
Convert format: 100% 1/1 [00:03<00:00,  3.69s/it]
warmup - generate
benchmark - generate
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: 

If you see `total throughput: XXX token/s` (in the hundreds), FlexGen is running on Colab.

## 4. Download the TA's pre-converted OPT-1.3B numpy weights

Downloads from the TA's Drive and extracts to `~/opt_weights/opt-1.3b-np/`. Takes ~2–5 minutes.

In [8]:
FILE_ID = "11iyGneldJ6M_z3FO0uR-yGQ4DL0udLA8"

import os
OPT_WEIGHTS_DIR = os.path.expanduser("~/opt_weights")
os.makedirs(OPT_WEIGHTS_DIR, exist_ok=True)

TAR_PATH = "/content/opt-1.3b-np.tar"
EXTRACTED_DIR = os.path.join(OPT_WEIGHTS_DIR, "opt-1.3b-np")

# 1. Download the tar (skip if already downloaded)
if not os.path.exists(TAR_PATH):
    print("Downloading from Google Drive...")
    !gdown "https://drive.google.com/uc?id={FILE_ID}" -O {TAR_PATH}
else:
    print(f"{TAR_PATH} already exists, skipping download.")

# 2. Verify download (size should be ~2.7 GB)
!ls -lh {TAR_PATH}

# 3. Extract to ~/opt_weights/ (creates ~/opt_weights/opt-1.3b-np/)
if not os.path.exists(os.path.join(EXTRACTED_DIR, "decoder.embed_positions.weight")):
    print(f"Extracting to {OPT_WEIGHTS_DIR}/ ...")
    !tar -xf {TAR_PATH} -C {OPT_WEIGHTS_DIR}
else:
    print(f"{EXTRACTED_DIR} already extracted, skipping.")

# 4. Verify the extracted directory
print("\n=== Extracted directory ===")
!du -sh {EXTRACTED_DIR}
!ls {EXTRACTED_DIR} | head -5
print(f"Total files: $(ls {EXTRACTED_DIR} | wc -l)")

Downloading...
From (original): https://drive.google.com/uc?id=11iyGneldJ6M_z3FO0uR-yGQ4DL0udLA8
From (redirected): https://drive.google.com/uc?id=11iyGneldJ6M_z3FO0uR-yGQ4DL0udLA8&confirm=t&uuid=83d2f395-c344-4718-81c8-c715b5f98f5a
To: /content/opt-1.3b-np.tar
100% 2.84G/2.84G [00:39<00:00, 71.5MB/s]
-rw------- 1 root root 2.7G May  9 11:47 /content/opt-1.3b-np.tar
Extracting to /root/opt_weights/ ...

=== Extracted directory ===
2.7G	/root/opt_weights/opt-1.3b-np
decoder.embed_positions.weight
decoder.embed_tokens.weight
decoder.layer_norm.bias
decoder.layer_norm.weight
decoder.layers.0.fc1.bias
Total files: $(ls /root/opt_weights/opt-1.3b-np | wc -l)


**Checkpoint**: you should now see `~/opt_weights/opt-1.3b-np/` (~2.7 GB) containing ~390 weight files.

## Q1. (30%) Progressive Offload Sweep

With `--gpu-batch-size 4 --prompt-len 128 --gen-len 16` fixed, run 5 weight distributions:

| Setting | `--percent` | Weight Distribution |
|---|---|---|
| (1) | `100 0 100 0 100 0` | 100% GPU |
| (2) | `50 50 100 0 100 0` | 50% GPU + 50% CPU |
| (3) | `50 0 100 0 100 0` | 50% GPU + 50% Disk |
| (4) | `0 50 100 0 100 0` | 50% CPU + 50% Disk |
| (5) | `0 0 100 0 100 0` | 100% Disk |

### Q1 dp(1): 100% GPU baseline

In [9]:
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 100 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    2>&1 | tee logs/q1_1.log

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.105 GB, hidden size (prefill): 0.002 GB
init weight...
warmup - generate
benchmark - generate
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------
3: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------

TorchDevice: cuda:0
  cur_mem: 2.6515 GB,  peak_mem: 2.8070 GB
TorchDevice: cpu
  cur_mem: 0.0000 GB,  peak_mem: 0.0000 GB
model size: 2.443 GB	cache size: 0.105 GB

### Q1 dp(2): 50% GPU + 50% CPU

In [10]:
# Mirror dp(1), changing --percent to 50 50 100 0 100 0 and tee'ing to logs/q1_2.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 50 50 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    2>&1 | tee logs/q1_2.log

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
/content/FlexLLMGen/flexllmgen/utils.py:132: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  data_ptr = tensor.storage().data_ptr()
<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.105 GB, hidden size (prefill): 0.002 GB
init weight...
warmup - generate
benchmark - generate
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
------------------------------------------------

### Q1 dp(3): 50% GPU + 50% Disk

In [11]:
# Mirror dp(1), changing --percent to 50 0 100 0 100 0 and tee'ing to logs/q1_3.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 50 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    2>&1 | tee logs/q1_3.log

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.105 GB, hidden size (prefill): 0.002 GB
init weight...
warmup - generate
benchmark - generate
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------
3: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------

TorchDevice: cuda:0
  cur_mem: 1.3341 GB,  peak_mem: 1.7134 GB
TorchDevice: cpu
  cur_mem: 0.0000 GB,  peak_mem: 0.0000 GB
model size: 2.443 GB	cache size: 0.105 GB

### Q1 dp(4): 50% CPU + 50% Disk

In [12]:
# Mirror dp(1), changing --percent to 0 50 100 0 100 0 and tee'ing to logs/q1_4.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 0 50 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    2>&1 | tee logs/q1_4.log

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
/content/FlexLLMGen/flexllmgen/utils.py:132: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  data_ptr = tensor.storage().data_ptr()
<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.105 GB, hidden size (prefill): 0.002 GB
init weight...
warmup - generate
benchmark - generate
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
------------------------------------------------

### Q1 dp(5): 100% Disk

In [13]:
# Mirror dp(1), changing --percent to 0 0 100 0 100 0 and tee'ing to logs/q1_5.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 0 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    2>&1 | tee logs/q1_5.log

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.105 GB, hidden size (prefill): 0.002 GB
init weight...
warmup - generate
benchmark - generate
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------
3: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------

TorchDevice: cuda:0
  cur_mem: 0.0089 GB,  peak_mem: 0.6197 GB
TorchDevice: cpu
  cur_mem: 0.0000 GB,  peak_mem: 0.0000 GB
model size: 2.443 GB	cache size: 0.105 GB

## Q2. (20%) Batch Size and I/O Amortization

With 100% disk offload (Q1 dp(5)) fixed, sweep `--gpu-batch-size = 1, 4, 16`.

> `--gpu-batch-size N` means FlexGen processes N independent prompts at once. The prompt content doesn't matter for this assignment; only the throughput numbers do.

### Q2 batch=1

In [14]:
# Mirror Q1 dp(5), adding --gpu-batch-size 1 and tee'ing to logs/q2_b1.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 0 0 100 0 100 0 \
    --gpu-batch-size 1 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    2>&1 | tee logs/q2_b1.log

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.026 GB, hidden size (prefill): 0.001 GB
init weight...
warmup - generate
benchmark - generate
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------

TorchDevice: cuda:0
  cur_mem: 0.0089 GB,  peak_mem: 0.4895 GB
TorchDevice: cpu
  cur_mem: 0.0000 GB,  peak_mem: 0.0000 GB
model size: 2.443 GB	cache size: 0.026 GB

### Q2 batch=4 (reuse Q1 dp(5) data — no need to rerun)

### Q2 batch=16

In [15]:
# Mirror Q1 dp(5), adding --gpu-batch-size 16 and tee'ing to logs/q2_b16.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 0 0 100 0 100 0 \
    --gpu-batch-size 16 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    2>&1 | tee logs/q2_b16.log

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.422 GB, hidden size (prefill): 0.009 GB
init weight...
warmup - generate
benchmark - generate
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------
15: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------

TorchDevice: cuda:0
  cur_mem: 0.0089 GB,  peak_mem: 1.0510 GB
TorchDevice: cpu
  cur_mem: 0.0000 GB,  peak_mem: 0.0000 GB
model size: 2.443 GB	cache size: 0.422 G

## Q3. (20%) Weight Compression

The `--compress-weight` flag quantizes weights from FP16 (16-bit) to 4-bit (4× compression ratio); they are dynamically de-quantized back to FP16 during loading to feed the GPU. Compare two scenarios with this flag enabled: (α) 100% GPU baseline, (β) 100% Disk offload.

Use Q1 dp(1) and dp(5) as the no-compression reference.

### Q3 (α): 100% GPU + compress

In [16]:
# Mirror Q1 dp(1), adding --compress-weight at the end and tee'ing to logs/q3_alpha.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 100 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    --compress-weight \
    2>&1 | tee logs/q3_alpha.log

/content/FlexLLMGen/flexllmgen/pytorch_backend.py:847: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  src = src.data[src_indices] if src_indices else src.data
/content/FlexLLMGen/flexllmgen/pytorch_backend.py:848: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  dst = dst.data[dst_indices] if dst_indices else dst.data
/con

### Q3 (β): 100% Disk + compress

In [17]:
# Mirror Q1 dp(5), adding --compress-weight at the end and tee'ing to logs/q3_beta.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 0 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    --compress-weight \
    2>&1 | tee logs/q3_beta.log

/content/FlexLLMGen/flexllmgen/pytorch_backend.py:875: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return data[indices] if indices else data
/content/FlexLLMGen/flexllmgen/compression.py:199: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  data = data.view(flatten_shape)[indices].contiguous()
/usr/local/lib/python3.12/

## Q4. (30%) I/O Behavior and Bottleneck Analysis under Disk Offload

For 100% Disk Offload (Q1 dp(5)), do two things: **Part 1** uses `iostat` to sample disk behavior; **Part 2** uses FlexGen's `--debug-mode breakdown` to measure stage timings.

### Install sysstat (Colab does not pre-install iostat)

In [18]:
!apt-get install -y -qq sysstat
!iostat -V | head -1

Preconfiguring packages ...
Selecting previously unselected package sysstat.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack .../sysstat_12.5.2-2ubuntu0.2_amd64.deb ...
Unpacking sysstat (12.5.2-2ubuntu0.2) ...
Setting up sysstat (12.5.2-2ubuntu0.2) ...

Creating config file /etc/default/sysstat with new version
update-alternatives: using /usr/bin/sar.sysstat to provide /usr/bin/sar (sar) in auto mode
Created symlink /etc/systemd/system/sysstat.service.wants/sysstat-collect.timer → /lib/systemd/system/sysstat-collect.timer.
Created symlink /etc/systemd/system/sysstat.service.wants/sysstat-summary.timer → /lib/systemd/system/sysstat-summary.timer.
Created symlink /etc/systemd/system/multi-user.target.wants/sysstat.service → /lib/systemd/system/sysstat.service.
Processing triggers for man-db (2.10.2-1) ...
sysstat version 12.5.2


### Q4 Part 1: Sample disk I/O

Colab does not allow multiple terminals, so the cell below uses `%%bash` to run `iostat` / `free` in the background, FlexGen in the foreground, then kills the samplers.

In [19]:
%%bash
# 1. Sample iostat and free in the background, remember the PIDs
iostat -x 1 > logs/q4_1_io_iostat.log &
IOSTAT_PID=$!
free -m -s 1 > logs/q4_1_io_free.log &
FREE_PID=$!

# 2. Run disk offload in the foreground (same setup as Q1 dp(5))
python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 0 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload 2>&1 | tee logs/q4_1_run.log

# 3. After FlexGen finishes, wait a few seconds for iostat to flush, then kill the samplers
sleep 5
kill -TERM $IOSTAT_PID $FREE_PID 2>/dev/null
wait 2>/dev/null
echo "Done. logs/q4_1_io_iostat.log $(wc -l < logs/q4_1_io_iostat.log) lines"

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.105 GB, hidden size (prefill): 0.002 GB
init weight...
warmup - generate
benchmark - generate
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------
3: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------

TorchDevice: cuda:0
  cur_mem: 0.0089 GB,  peak_mem: 0.5884 GB
TorchDevice: cpu
  cur_mem: 0.0000 GB,  peak_mem: 0.0000 GB
model size: 2.443 GB	cache size: 0.105 GB

### Extract the 4 columns from the iostat log

Colab's disk name is `sda`. The awk below is ready to run.

In [20]:
%%bash
awk '$1=="sda" {
    if ($2+0 > r) r=$2+0          # r/s peak
    if ($3+0 > k) k=$3+0          # rkB/s peak
    if ($7+0 > sz) sz=$7+0        # rareq-sz peak (max across samples)
    if ($NF+0 > u) u=$NF+0        # %util peak
} END {
    printf "r/s peak       = %s\n", r
    printf "rkB/s peak     = %s\n", k
    printf "rareq-sz peak  = %.2f\n", sz
    printf "%%util peak     = %s\n", u
}' logs/q4_1_io_iostat.log

r/s peak       = 2583
rkB/s peak     = 255976
rareq-sz peak  = 128.00
%util peak     = 80.8


### Q4 Part 2: Run `--debug-mode breakdown` for both scenarios

> `--debug-mode breakdown` only runs 20 batches to collect timing; **do not compare these throughput numbers to Q1**. Look only at `load_weight` and `compute_layer_decoding`. The output unit is **seconds**; values can differ by several orders of magnitude across scenarios, so keep at least 6 decimal places.

### Part 2 baseline (100% GPU)

In [21]:
# Mirror Q1 dp(1), adding --debug-mode breakdown and tee'ing to logs/q4_2_baseline.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 100 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    --debug-mode breakdown \
    2>&1 | tee logs/q4_2_baseline.log

<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.105 GB, hidden size (prefill): 0.002 GB
init weight...
warmup - generate
benchmark - generate
100%|██████████| 20/20 [00:00<00:00, 73.75it/s]
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
#layers: 50
#batches prefill:  50
#batches decoding: 750
load_weight            (per-layer): 0.000036 s
load_cache_prefill     (per-batch): 0.000010 s
store_cache_prefill    (per-batch): 0.000124 s
compute_layer_prefill  (per-batch): 0.002109 s
load_cache_decoding    (per-batch): 0.000035 s
store_cache_decoding   (per-batch): 0.000094 s
compute_layer_decoding (per-batch): 0.006474 s
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France
----------------------------------------------------------

### Part 2 disk offload (100% Disk)

In [22]:
# Mirror Q1 dp(5), adding --debug-mode breakdown and tee'ing to logs/q4_2_disk.log
!python -m flexllmgen.flex_opt \
    --model facebook/opt-1.3b \
    --percent 0 0 100 0 100 0 \
    --gpu-batch-size 4 \
    --prompt-len 128 --gen-len 16 \
    --offload-dir flexgen_offload \
    2>&1 | tee logs/q4_2_disk.log

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1146: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
<run_flexllmgen>: args.model: facebook/opt-1.3b
model size: 2.443 GB, cache size: 0.105 GB, hidden size (prefill): 0.002 GB
init weight...
warmup - generate
benchmark - generate
Outputs:
----------------------------------------------------------------------
0: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------
3: Paris is the capital city of France. It is the most populous city in France, with an estimated population of
----------------------------------------------------------------------

TorchDevice: cuda:0
  cur_mem: 0.0089 GB,  peak_mem: 0.5884 GB
TorchDevice: cpu
  cur_mem: 0.0000 GB,  peak_mem: 0.0000 GB
model size: 2.443 GB	cache size: 0.105 GB